In [3]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.medgan.models import MEDGAN
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = Path.cwd().parents[1]

raw_file = ROOT / "datasets" / "nursery.csv"
processed_file = ROOT / "preprocessed_data" / "nursery.csv"
artifact_dir = ROOT / "artifacts"

print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file)
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=MEDGAN(
        ae_pretrain_epochs=2,
        gan_epochs=2
    )
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="nursery",
    artifact_store=store,
    model_name="medgan",
)

print(results)

Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\nursery.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\nursery.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\nursery.csv


C:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\pipeline\train_test_split\pipeline.py:270: UserWarning: Registered dataset 'nursery' column schema differs from 'c:\\Users\\savin\\OneDrive\\Desktop\\Semester 02\\Capstone Project\\Katabatic\\katabatic\\preprocessed_data\\nursery.csv'
  reg.register_if_absent(dataset_name, profile_csv, target_column=target_column)


Loaded data with shape: (12960, 9)


INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Training MedGAN Model
INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Loaded training data: (10368, 8)
INFO:katabatic.models.medgan.models:Categorical columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8']
INFO:katabatic.models.medgan.models:Continuous columns: []
INFO:katabatic.models.medgan.models:Data normalized to [0, 1] range
INFO:katabatic.models.medgan.models:Original range: [0.00, 4.00]
INFO:katabatic.models.medgan.models:Normalized range: [0.00, 1.00]
INFO:katabatic.models.medgan.models:
Phase 1: Pretraining Autoencoder for 2 epochs...
INFO:katabatic.models.medgan.models:Epoch 1/2: AE Loss = 0.675489
INFO:katabatic.models.medgan.models:Epoch 2/2: AE Loss = 0.633192
INFO:katabatic.models.medgan.models:
Phase 2: Training GAN 

Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved dataset artifact under datasets/nursery/split-20260830-203620


INFO:katabatic.models.medgan.models:Epoch 1/2: D Loss = 1.297501, G Loss = 0.725753
INFO:katabatic.models.medgan.models:Epoch 2/2: D Loss = 0.789434, G Loss = 1.238046
INFO:katabatic.models.medgan.models:
Generating 10368 synthetic samples...
INFO:katabatic.models.medgan.models:Adding one existing training sample for missing target classes...
INFO:katabatic.models.medgan.models:
Synthetic data saved to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\models\medgan_nursery_train-20260830-203621\synthetic
INFO:katabatic.models.medgan.models:Training complete!


{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(dataset_name='nursery', dataset_version='split-20260830-203620'), 'model_ref': ModelRef(model_name='medgan', dataset_name='nursery', dataset_version='split-20260830-203620', train_run_id='train-20260830-203621'), 'evaluation_refs': []}


In [5]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "nursery"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = train_df.columns[-1]

categorical_cols = train_df.columns[:-1].tolist()
continuous_cols = []

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Synthetic type:", type(synthetic_df))
print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\nursery\split-20260830-203620
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Synthetic sample:
            0            1       2  3         4           5            6  \
0  great_pret  less_proper  foster  2  critical  convenient  problematic   
1  great_pret     improper  foster  2  critical      inconv  problematic   
2  great_pret     improper  foster  2  critical  convenient      nonprob   
3  great_pret     improper  foster  1  critical  convenient  problematic   
4  great_pret     improper  foster  2  critical      inconv      nonprob   

          7         8  
0  priority  priority  
1  priority  priority  
2  priority  priority  
3  priority  priority  
4  priority  priority  

Running fidelity evaluation...

=== Fidelity Evaluation ===
Overall fidelity score: 0.5417

Categorical JSD (lower = better)  ->  score: 0.5417
  0                              JSD =

c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\savin\O


=== Utility Evaluation ===
Overall utility score: 0.3503

Classifier   Metric     TSTR mean    TRTR mean    Delta   
--------------------------------------------------------
LR           accuracy   0.3220       0.7617       0.4397
LR           f1         0.1928       0.7545       0.5617
DT           accuracy   0.3281       0.9904       0.6623
DT           f1         0.1644       0.9906       0.8262
RF           accuracy   0.3287       0.9766       0.6479
RF           f1         0.1633       0.9762       0.8129
LinearSVM    accuracy   0.2679       0.7505       0.4826
LinearSVM    f1         0.1786       0.7423       0.5637
MLP          accuracy   0.2991       0.9984       0.6993
MLP          f1         0.1976       0.9986       0.801

Running diversity evaluation...

=== Diversity Evaluation ===
Overall diversity score: 0.6902

Category Coverage (% of real categories in synth)
  0                              66.7%
  1                              80.0%
  2                             